# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1, 1.33 : isna() делает boolean список, если None то True, sum() возвращает кол-во тру значений, у нас 1. fillna(0) заполняет все None 0, поэтому у нас (1 + 0 + 3) / 3 = 1.33
2. 2, наши " a" и "A " приведутся в одну форму, так как уберутся пробелы .strip() и А станет а через .lower(), так что уникальных останется только 2
3. 1, len нам возвращает кол-во ключей, и drop_duplicates убирает идентичные строки, а не элементы в списке
4. 10005, потому что + выполняется на строку, а не на число, потом уже строка превратится в число
5. [9, 10] , вернется лист из двух месяцев, 9 и 10

In [1]:
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

1 1.3333333333333333
2
3
1005
[9, 10]


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [14]:
# 1
import pandas as pd
import numpy as np
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("Проверка amount:",df["amount"].isna().sum())    # проверка: у amount нет NaN (0)
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("Проверка city:",df["city"].isna().sum())    # проверка: у city нет NaN (0)
    df = df.drop_duplicates()
    print("После удаления дублей:",len(df))
    not_test = df["customer_id"] != "TEST"
    df = df[not_test]
    print("После удаления ТЕСТ:",len(df))
    return df

df_clean = clean("orders.csv")
df_clean


       

Проверка amount: 0
Проверка city: 0
После удаления дублей: 5283
После удаления ТЕСТ: 5280


,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00,C00557,Астана,Электроника,22150.0,NaN,оплачен
5461,103460,2026-10-31 22:00,C02162,Алматы,Электроника,24670.0,2.0,отменён


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [27]:
# 2
sep = df_clean["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df_clean["status"] == "оплачен"

print("Сентябрь:",df_clean[paid & sep]["amount"].sum())
print("Октябрь:",df_clean[paid & octo]["amount"].sum())
print("Изменение в %:", (df_clean[paid & octo]["amount"].sum() / df_clean[paid & sep]["amount"].sum() - 1) * 100 , "%")

cities = df_clean["city"].unique()
for c in cities:
    print("")
    print(c)
    print("Сентябрь:",df_clean[(df_clean["city"] == c) & paid & sep]["amount"].sum())
    print("Октябрь:",df_clean[(df_clean["city"] == c) & paid & octo]["amount"].sum())
    print("Изменение в %:", (df_clean[(df_clean["city"] == c) & paid & octo]["amount"].sum() / df_clean[(df_clean["city"] == c) & paid & sep]["amount"].sum() - 1) * 100 , "%")
    

Сентябрь: 35243410.0
Октябрь: 34419520.0
Изменение в %: -2.337713632137184 %

Алматы
Сентябрь: 14501070.0
Октябрь: 14643020.0
Изменение в %: 0.9788932816681806 %

Астана
Сентябрь: 9034220.0
Октябрь: 8850530.0
Изменение в %: -2.0332690591993607 %

Караганда
Сентябрь: 3648520.0
Октябрь: 3371680.0
Изменение в %: -7.587734204554176 %

Шымкент
Сентябрь: 4558650.0
Октябрь: 4044010.0
Изменение в %: -11.28930714136861 %

Атырау
Сентябрь: 923990.0
Октябрь: 969860.0
Изменение в %: 4.964339440903043 %

Актобе
Сентябрь: 2576960.0
Октябрь: 2540420.0
Изменение в %: -1.41794983236061 %


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Шаг | Строк | Изменение выручки окт/сент |
|---|---|---|
| Как считал менеджер (только оплаченные) | 0 | −22,7% |
| Без дублей | 180 |-19.59%|
| Без TEST | 3 | -2.3%|


Больше всего искажал цифру TEST, потому что у него была большая сумма чека

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [42]:
# 4
sum_days = df_clean["delivery_days"].isna().sum()
print("Пропусков:", sum_days)
print(df_clean.loc[df_clean["delivery_days"].isna(), "city"].value_counts())
print(df_clean.loc[df_clean["delivery_days"].isna(), "status"].value_counts())
shymkent = df_clean.loc[df_clean["city"] == "Шымкент", "delivery_days"]
shymkent_2_days = shymkent[shymkent < 3]
print("Доля доставки за 2 дня в Шымкенте:",len(shymkent_2_days) / len(shymkent) * 100 , "%")

Пропусков: 158
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
status
оплачен    144
отменён     14
Name: count, dtype: int64
Доля доставки за 2 дня в Шымкенте: 28.994845360824744 %


dropna() по столбцу убирает цифры из расчетов, ничего не придумываем но данных становится меньше, fillna(медиана) добавляет значения но мы их придумали, по факту значений нету, а fillna(0) добавляет нули, хотя доставка в среднем больше чем 0 дней. Поэтому самый честный вариант это убрать цифры из расчетов и рассчитать из того что есть, а fillna(0) вводит в заблуждение из за того что мы ставим доставку за 0 дней, что очень маловероятно.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [43]:
# 5
df = pd.read_csv("orders.csv")
df = df.drop_duplicates()
df["city"] = df["city"].str.strip().str.lower().map(CITY)
print(df["city"].nunique(), len(df))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [60]:
# 6
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"])
day_names = df_clean["order_date"].dt.day_name().value_counts()
print(day_names)
print("Меньше всего заказов в Понедельник, меньше на:", (day_names.max() / day_names.min() - 1) * 100, "% чем в Четверг. В понедельник привлекаем меньше курьеров, а в четверг наоборот по максмуму")


order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Меньше всего заказов в Понедельник, меньше на: 14.912280701754387 % чем в Четверг. В понедельник привлекаем меньше курьеров, а в четверг наоборот по максмуму


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [79]:
# 7
shymkent_sep = shymkent[sep]
shymkent_octo = shymkent[octo]
print("Средний срок доставки в Шымкенте в сентябре", shymkent_sep.mean())
print("Средний срок доставки в Шымкенте в сентябре", shymkent_octo.mean())
print(df_clean[(df_clean["city"] == "Шымкент") & (df_clean["delivery_days"] > 4)]["order_date"].sort_values())
for c in cities:
    print("")
    print(c)
    print(df_clean[(df_clean["city"] == c) & (df_clean["delivery_days"] > 4)]["order_date"].sort_values())

Средний срок доставки в Шымкенте в сентябре 2.031518624641834
Средний срок доставки в Шымкенте в сентябре 6.5358024691358025
2793   2026-10-01 10:00:00
2813   2026-10-01 15:00:00
2816   2026-10-01 15:00:00
2817   2026-10-01 15:00:00
2818   2026-10-01 15:00:00
               ...        
5397   2026-10-31 10:00:00
5409   2026-10-31 12:00:00
5425   2026-10-31 15:00:00
5441   2026-10-31 17:00:00
5448   2026-10-31 18:00:00
Name: order_date, Length: 405, dtype: datetime64[us]

Алматы
Series([], Name: order_date, dtype: datetime64[us])

Астана
Series([], Name: order_date, dtype: datetime64[us])

Караганда
Series([], Name: order_date, dtype: datetime64[us])

Шымкент
2793   2026-10-01 10:00:00
2813   2026-10-01 15:00:00
2816   2026-10-01 15:00:00
2817   2026-10-01 15:00:00
2818   2026-10-01 15:00:00
               ...        
5397   2026-10-31 10:00:00
5409   2026-10-31 12:00:00
5425   2026-10-31 15:00:00
5441   2026-10-31 17:00:00
5448   2026-10-31 18:00:00
Name: order_date, Length: 405, dtype

Сообщение менеджеру: с 1 октября 2026 г. в городе Шымкент срок доставки в 405 заказах был 5 или больше. Стоит проверить почему срок доставки увеличился, может быть стоит проверить работу курьеров или нагруженность в октябре.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [97]:
# 8
def to_amount(ser):
    ser = pd.to_numeric(ser.str.replace(" ", "").str.replace(",", "."), errors="coerce")
    return ser

ser = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print(ser)
ser = to_amount(ser)
print(ser, ser.sum())
df = pd.read_csv("orders.csv")
df["amount"] = to_amount(df["amount"])
print(df["amount"].sum())

0        15 990
1       15990.0
2      1 200,50
3    нет данных
dtype: str
0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64 33180.5
109883360.0
